# AI-Assisted Defect Detection — Train & Use the Model

This notebook (1) converts **MVTec AD** into a YOLO segmentation dataset, (2) fine-tunes an **Ultralytics YOLO** segmentation model,
(3) evaluates it, (4) calibrates the CIELAB colour reference, and (5) **uses the trained model in code** to give **PASS / FAIL**
on images, an uploaded video, or a camera.

Works locally (Jupyter/VS Code) or in **Google Colab** (Runtime → Change runtime type → GPU).
Set `DEMO_MODE = True` in the config cell to smoke-test everything on a tiny synthetic dataset in a few minutes.

## 1. Setup

In [ ]:
import sys, os, subprocess, shutil, zipfile, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q ultralytics opencv-python-headless
else:
    !pip install -q ultralytics opencv-python matplotlib

# --- locate the project folder (the one containing pipeline.py) ---
def find_project():
    for p in [Path.cwd(), Path.cwd() / "defect_inspection", Path.cwd().parent,
              Path("/content/defect_inspection")]:
        if (p / "pipeline.py").exists():
            return p
    return None

PROJECT = find_project()
if PROJECT is None and IN_COLAB:
    from google.colab import files
    print("Upload defect_inspection.zip")
    up = files.upload()
    zname = next(iter(up))
    zipfile.ZipFile(zname).extractall("/content")
    PROJECT = find_project()
assert PROJECT is not None, "Put this notebook inside the defect_inspection folder (next to pipeline.py)."

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("Project:", PROJECT)

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (training will be slow)")

## 2. Configuration

In [ ]:
DEMO_MODE   = False   # True = tiny synthetic dataset, 20 epochs: only checks that the code works

# Path to the extracted MVTec AD folder (it contains bottle/, screw/, ...). Ignored when DEMO_MODE.
MVTEC_DIR   = "/content/mvtec_anomaly_detection" if IN_COLAB else str(Path.home() / "datasets" / "mvtec_anomaly_detection")
CATEGORIES  = ["bottle", "metal_nut", "screw", "capsule", "tile"]

MODEL       = "yolo11n-seg.pt"   # yolo11s-seg.pt = more accurate, slower
EPOCHS      = 100
IMGSZ       = 640                # try 800-1024 for small defects (screw, capsule)
BATCH       = 16
DEVICE      = 0 if torch.cuda.is_available() else "cpu"

if DEMO_MODE:
    EPOCHS, IMGSZ, BATCH = 20, 256, 8
    CATEGORIES = ["widget", "gizmo"]
    MVTEC_DIR  = str(PROJECT / "datasets" / "fake_mvtec")

DATA_DIR = PROJECT / "datasets" / ("fake_yolo" if DEMO_MODE else "mvtec_yolo")
WEIGHTS  = PROJECT / "models" / "best.pt"

## 3. Get the MVTec AD dataset
MVTec AD is free for non-commercial use but needs a licence acceptance, so download it yourself from
<https://www.mvtec.com/company/research/datasets/mvtec-ad> and extract it to `MVTEC_DIR`
(in Colab: upload the archive to Google Drive, mount it and extract, or upload it to `/content`).
The cell below checks the folder; with `DEMO_MODE` it builds a synthetic stand-in instead.

In [ ]:
if DEMO_MODE:
    sys.path.insert(0, str(PROJECT / "tests"))
    from make_fake_mvtec import build
    build(MVTEC_DIR, categories=tuple(CATEGORIES), n_train=30, n_test_good=10, n_def=20)
    print("Synthetic dataset built at", MVTEC_DIR)

# --- Colab + Google Drive example (edit the paths, then uncomment) ---
# from google.colab import drive; drive.mount('/content/drive')
# !tar -xf "/content/drive/MyDrive/mvtec_anomaly_detection.tar.xz" -C /content
# (the official download is a .tar.xz; it extracts to a folder named mvtec_anomaly_detection)

missing = [c for c in CATEGORIES if not (Path(MVTEC_DIR) / c / "test").is_dir()]
assert not missing, f"Not found under {MVTEC_DIR}: {missing}. Check MVTEC_DIR."
print("MVTec folder OK:", MVTEC_DIR)

## 4. Convert masks → YOLO segmentation labels
Defective images + ground-truth masks → polygon labels (single class `0 = defect`). Defect-free images are added as background
samples. Splits are stratified by defect type. *(This is a supervised segmentation experiment derived from MVTec annotations,
not the original anomaly-detection protocol.)*

In [ ]:
extra = ["--train-good", "25", "--extra-eval-good", "6"] if DEMO_MODE else []
cmd = [sys.executable, "prepare_dataset.py", "--mvtec", MVTEC_DIR, "--out", str(DATA_DIR),
       "--categories", *CATEGORIES, "--clean", *extra]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2500:] or r.stderr[-2500:])
assert (DATA_DIR / "data.yaml").exists(), "Dataset conversion failed"

In [ ]:
# Quick visual check: ground-truth polygons drawn on a few training images
import cv2, glob, random, numpy as np
import matplotlib.pyplot as plt

imgs = [p for p in glob.glob(str(DATA_DIR / "images" / "train" / "*")) if "_good_" not in p]
random.seed(0)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, p in zip(axes, random.sample(imgs, min(4, len(imgs)))):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    lab = Path(p.replace("images", "labels")).with_suffix(".txt")
    for line in lab.read_text().splitlines():
        v = np.array(line.split()[1:], float).reshape(-1, 2) * [w, h]
        cv2.polylines(im, [v.astype(np.int32)], True, (255, 0, 0), max(2, w // 300))
    ax.imshow(im); ax.set_title(Path(p).stem[:28], fontsize=8); ax.axis("off")
plt.show()

## 5. Train
Fine-tunes the pretrained segmentation model. Flip augmentation is **off** (a mirrored `metal_nut` is itself a defect in MVTec) and
hue augmentation is off (it would fight the separate colour-inspection branch).

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=str(DATA_DIR / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    patience=30, device=DEVICE, workers=2 if not IN_COLAB else 2,
    project=str(PROJECT / "runs"), name="defect_seg", exist_ok=True, seed=0, plots=True,
    fliplr=0.0, flipud=0.0, hsv_h=0.0,
)

best = PROJECT / "runs" / "defect_seg" / "weights" / "best.pt"
WEIGHTS.parent.mkdir(exist_ok=True)
shutil.copy2(best, WEIGHTS)
print("Trained weights saved to", WEIGHTS)

In [ ]:
from IPython.display import Image, display
for name in ["results.png", "confusion_matrix.png"]:
    f = PROJECT / "runs" / "defect_seg" / name
    if f.exists():
        display(Image(filename=str(f), width=900))

## 6. Colour calibration (CIELAB / CIEDE2000)
Builds a reference colour and ΔE₀₀ threshold per category from **defect-free** training images.
For your own physical product, calibrate from your own good samples under your camera and lighting (see the commented example). Done before evaluation so the reported PASS/FAIL includes the colour branch.

In [ ]:
r = subprocess.run([sys.executable, "calibrate_color.py", "--mvtec", MVTEC_DIR, "--categories", *CATEGORIES],
                   capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])

# For your own product instead (folder/video/camera of GOOD parts):
# !python calibrate_color.py --source good_samples/ --name mypart

## 7. Evaluate on the held-out test split
Reports per-category PASS/FAIL accuracy, recall (defect detection rate), false-positive / false-negative rate, pixel IoU, latency, and
recommends a confidence / min-area operating point. **Pick thresholds on `--split val`, then report once on `--split test`.**

In [ ]:
# 1) choose thresholds on the validation split
r = subprocess.run([sys.executable, "evaluate.py", "--data-dir", str(DATA_DIR), "--weights", str(WEIGHTS),
                    "--split", "val", "--imgsz", str(IMGSZ), "--device", str(DEVICE),
                    "--out", "outputs/eval_val", "--skip-ultralytics-val"], capture_output=True, text=True)
print(r.stdout[-1800:] or r.stderr[-1800:])

In [ ]:
rep = json.load(open("outputs/eval_val/report.json"))
REC = rep["recommended_operating_point"]
CONF, MIN_AREA = REC["conf"], REC["min_area_pct"]
print("Chosen on validation ->", REC)

# 2) final numbers on the test split with those thresholds
r = subprocess.run([sys.executable, "evaluate.py", "--data-dir", str(DATA_DIR), "--weights", str(WEIGHTS),
                    "--split", "test", "--imgsz", str(IMGSZ), "--device", str(DEVICE),
                    "--conf", str(CONF), "--min-area", str(MIN_AREA), "--out", "outputs/eval_test"],
                   capture_output=True, text=True)
print(r.stdout[-1800:] or r.stderr[-1800:])

## 8. Use the trained model in code
`InspectionPipeline.inspect(frame)` runs YOLO segmentation + colour analysis + decision engine and returns the PASS/FAIL record.

In [ ]:
from color_analysis import load_profiles
from config import InspectionConfig, TEXTURE_CATEGORIES, COLOR_REFERENCE_PATH
from pipeline import InspectionPipeline

profiles = load_profiles(COLOR_REFERENCE_PATH)

def make_pipeline(category=None):
    # texture categories fill the whole frame -> use the full image as the product region
    roi = "full" if category in TEXTURE_CATEGORIES else "auto"
    cfg = InspectionConfig(imgsz=IMGSZ, conf=CONF, min_area_pct=MIN_AREA, roi_mode=roi,
                           require_product=(roi == "auto"), device=DEVICE)
    return InspectionPipeline(WEIGHTS, cfg, profiles.get(category))

def inspect_image(path, category=None, show=True):
    frame = cv2.imread(str(path))
    pipe = make_pipeline(category)
    insp = pipe.inspect(frame, Path(path).stem)
    rec = insp.record
    if show:
        vis = pipe.annotate(frame, insp, rec["final_result"])
        plt.figure(figsize=(6, 6)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
    return rec

# try one defective and one good test image (category comes from the dataset manifest)
import csv
man = [r for r in csv.DictReader(open(DATA_DIR / "manifest.csv")) if r["split"] == "test"]
for label in ("1", "0"):
    r = next(r for r in man if r["label"] == label)
    print("Ground truth:", "FAIL (defective)" if label == "1" else "PASS (good)", "|", r["image"])
    print(json.dumps(inspect_image(DATA_DIR / r["image"], r["category"]), indent=2))

In [ ]:
# PASS/FAIL for every held-out test image, compared with the ground truth
import collections
rows = [r for r in csv.DictReader(open(DATA_DIR / "manifest.csv")) if r["split"] == "test"]
pipes, results = {}, []
for r in rows:
    cat = r["category"]
    pipes.setdefault(cat, make_pipeline(cat))
    frame = cv2.imread(str(DATA_DIR / r["image"]))
    out = pipes[cat].inspect(frame, Path(r["image"]).stem).record["final_result"]
    results.append((cat, "FAIL" if r["label"] == "1" else "PASS", out))

cm = collections.Counter((truth, pred) for _, truth, pred in results)
acc = sum(t == p for _, t, p in results) / len(results)
print(f"{len(results)} test images | accuracy {acc:.3f}")
print("truth/pred   PASS  FAIL")
for t in ("PASS", "FAIL"):
    print(f"{t:<11} {cm[(t,'PASS')]:>5} {cm[(t,'FAIL')]:>5}")

## 9. Test with an uploaded video or the camera
**Uploaded video** — run the cell, choose a file; you get an annotated video, a CSV log and an overall PASS/FAIL.

In [ ]:
VIDEO_PATH = None          # set a path here, or leave None to upload (Colab) / skip
CATEGORY   = None          # e.g. "bottle" to use that category's colour profile; None = no colour check

if VIDEO_PATH is None and IN_COLAB:
    from google.colab import files
    VIDEO_PATH = next(iter(files.upload()))

if VIDEO_PATH:
    args = [sys.executable, "inspector.py", "--source", str(VIDEO_PATH), "--weights", str(WEIGHTS),
            "--imgsz", str(IMGSZ), "--conf", str(CONF), "--min-area", str(MIN_AREA),
            "--device", str(DEVICE), "--no-display", "--out", "outputs/video"]
    if CATEGORY:
        args += ["--color-profile", CATEGORY] + (["--roi", "full"] if CATEGORY in TEXTURE_CATEGORIES else [])
    else:
        args += ["--no-color"]
    r = subprocess.run(args, capture_output=True, text=True)
    print(r.stdout[-1500:] or r.stderr[-1500:])
    print(open(next(Path("outputs/video").glob("*/summary.json"))).read())
else:
    print("No video selected - skipping.")

**Live camera** — a notebook cannot open a window on your webcam in Colab, so run this on your own computer after downloading `models/best.pt`
(and `models/color_reference.json` if you want the colour check):
```bash
python inspector.py --source 0                         # q = quit, s = snapshot, space = pause
python inspector.py --source 0 --color-profile mypart  # with your calibrated colour reference
```
Locally in Jupyter you can also grab a single camera frame in code:
```python
cap = cv2.VideoCapture(0); ok, frame = cap.read(); cap.release()
pipe = make_pipeline()
print(pipe.inspect(frame).record["final_result"])
```

## 10. Save / download the trained model

In [ ]:
print("Weights:", WEIGHTS, f"({WEIGHTS.stat().st_size/1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(WEIGHTS))
    if COLOR_REFERENCE_PATH.exists():
        files.download(str(COLOR_REFERENCE_PATH))